# Exercise 2.2.2: Data Types and Subsetting
*Exercício 2.2.2: Tipos de Dados e Subconjuntos*

This notebook continues with `datania_households_raw.csv` from 2.2.1. Using **only the tools from Lesson 2.2**, you fix data types and save a *typed checkpoint* that the cleaning step (2.2.3) will pick up.

*Este notebook continua com `datania_households_raw.csv` do 2.2.1. Usando **apenas as ferramentas da Lição 2.2**, vai corrigir os tipos de dados e guardar um *ponto de controlo com tipos corrigidos* que o passo de limpeza (2.2.3) irá utilizar.*

You will practice:
- Telling a **DataFrame** from a **Series** and checking dtypes
- Standardising text and fixing categories with `.str` methods and `.replace()`
- Converting messy text to numbers with `.str.replace()` + `pd.to_numeric()`
- Converting text to dates with `pd.to_datetime()` and the `.dt` accessor
- Subsetting (boolean indexing, `isin()`, `str.contains()`) to *inspect* problems
- Saving a typed checkpoint to `10_cleaned/`

*Vai praticar:*
- *Distinguir um **DataFrame** de uma **Series** e verificar os dtypes*
- *Uniformizar texto e corrigir categorias com métodos `.str` e `.replace()`*
- *Converter texto desorganizado em números com `.str.replace()` + `pd.to_numeric()`*
- *Converter texto em datas com `pd.to_datetime()` e o acessor `.dt`*
- *Criar subconjuntos (indexação booleana, `isin()`, `str.contains()`) para *inspecionar* problemas*
- *Guardar um ponto de controlo com tipos corrigidos em `10_cleaned/`*

> **Pipeline:** reads `0_raw/` and writes a typed file to `10_cleaned/`. Exercise 2.2.3 reads that file.

> ***Fluxo de trabalho:** lê de `0_raw/` e escreve um ficheiro com tipos corrigidos em `10_cleaned/`. O Exercício 2.2.3 lê esse ficheiro.*

### Path Setup (run first)
*Configuração do caminho (execute primeiro)*

In [ ]:
import os
import numpy as np
import pandas as pd

DATA_RAW_DIR = # your code here | o seu código aqui
FILE_NAME = # your code here | o seu código aqui
raw_path = os.path.join(DATA_RAW_DIR, FILE_NAME)

# Identifiers and codes must stay as text (preserve leading zeros)
# Os identificadores e códigos devem manter-se como texto (preservar os zeros à esquerda)
df = pd.read_csv(raw_path, dtype={'hh_id': str, 'region_code': str})

print('Loaded:', df.shape)
df.head()

---

## Task 1: DataFrame vs Series
*Tarefa 1: DataFrame vs Series*

Selecting a single column returns a **Series**: a one-dimensional object with its own dtype and methods. Type checks (`.dtype`, `.str`, `.dt`, `.value_counts()`) operate on Series, not the whole DataFrame.

*Selecionar uma única coluna devolve uma **Series**: um objeto unidimensional com o seu próprio dtype e os seus métodos. As verificações de tipo (`.dtype`, `.str`, `.dt`, `.value_counts()`) funcionam sobre Series, não sobre o DataFrame inteiro.*

In [ ]:
# Compare the two object types | Comparar os dois tipos de objeto
print(type(df))
print(type(df['income_dkw']))

In [ ]:
# Print the dtype pandas assigned to each column
# Imprimir o dtype que o pandas atribuiu a cada coluna
df.  # your code here | o seu código aqui

**Questions:**

- What dtype did pandas assign to `income_dkw`? Is it numeric? Why?
- What dtype is `survey_date`? What must you do before extracting the month from it?
- `hh_id` and `region_code` are text. What would have happened to `region_code` without the `dtype=` argument in `read_csv`?

***Perguntas:***

- *Que dtype é que o pandas atribuiu a `income_dkw`? É numérico? Porquê?*
- *Qual é o dtype de `survey_date`? O que tem de fazer antes de extrair o mês?*
- *`hh_id` e `region_code` são texto. O que teria acontecido a `region_code` sem o argumento `dtype=` no `read_csv`?*

---

## Task 2: Standardise text and fix categories
*Tarefa 2: Uniformizar texto e corrigir categorias*

Two replace methods look similar but differ:

*Dois métodos de substituição parecem semelhantes mas são diferentes:*

- **`.replace({'old': 'new'})`** matches and replaces **whole values** (exact match)
- **`.str.replace('old', 'new')`** replaces a **substring** inside each string

- ***`.replace({'antigo': 'novo'})`** compara e substitui **valores inteiros** (correspondência exata)*
- ***`.str.replace('antigo', 'novo')`** substitui uma **subcadeia** dentro de cada texto*

Inspect the two categorical columns, then fix them with whole-value `.replace()`.

*Inspecione as duas colunas categóricas e corrija-as com `.replace()` sobre o valor inteiro.*

In [ ]:
# Inspect both categorical columns | Inspecionar as duas colunas categóricas
print(df['urban_rural'].value_counts(dropna=False))
print()
print(df['region_code'].value_counts(dropna=False))

In [ ]:
# Fix the typo 'Urbn' -> 'Urban' | Corrigir o erro de escrita 'Urbn' -> 'Urban'
df['urban_rural'] = df['urban_rural'].  # your code here: .replace('Urbn', 'Urban')
                                        # o seu código aqui: .replace('Urbn', 'Urban')

# region_code: '99' is a coded-missing value -> NaN; '1' should be the 2-digit '01'.
# Why NOT .str.replace('1','01')? It would turn '01'->'001', '21'->'021'. Use whole-value .replace.
# region_code: '99' é um código de valor em falta -> NaN; '1' deve ser '01' com dois dígitos.
# Porquê NÃO usar .str.replace('1','01')? Transformaria '01'->'001' e '21'->'021'. Use .replace sobre o valor inteiro.
df['region_code'] = df['region_code'].  # your code here: .replace({'99': np.nan, '1': '01'})
                                        # o seu código aqui: .replace({'99': np.nan, '1': '01'})

print(df['urban_rural'].value_counts(dropna=False))
print()
print(df['region_code'].value_counts(dropna=False))

**Questions:**

- Why would `.str.replace('1', '01')` corrupt codes like `'01'`, `'16'`, or `'21'`?
- After the fix, how many `NaN` values does `region_code` have, and where did each come from?

***Perguntas:***

- *Porque é que `.str.replace('1', '01')` corromperia códigos como `'01'`, `'16'` ou `'21'`?*
- *Depois da correção, quantos valores `NaN` tem `region_code` e de onde veio cada um?*

---

## Task 3: Convert text to numbers
*Tarefa 3: Converter texto em números*

`income_dkw` holds household income but arrives as text: spaces, commas, the `Ar` prefix, and labels like `unknown`. Remove the formatting with `.str.replace()`, replace the text labels (`unknown`, `NA`) with `NaN` using `.replace()`, then convert with `pd.to_numeric()`. Because every non-number is now `NaN`, use the **strict default** `errors='raise'`: if a label was missed the conversion fails loudly instead of hiding it.

*`income_dkw` contém o rendimento do agregado familiar, mas chega como texto: espaços, vírgulas, o prefixo `Ar` e rótulos como `unknown`. Retire a formatação com `.str.replace()`, substitua os rótulos de texto (`unknown`, `NA`) por `NaN` com `.replace()` e depois converta com `pd.to_numeric()`. Como todos os valores não numéricos são agora `NaN`, use o **valor por omissão estrito** `errors='raise'`: se algum rótulo tiver escapado, a conversão falha de forma visível em vez de esconder o problema.*

In [ ]:
# Inspect the distinct raw values first
# Comece por inspecionar os valores brutos distintos
df['income_dkw'].unique()

In [ ]:
# Remove formatting, then turn the text labels into NaN so the column is fully numeric
# Retirar a formatação e transformar os rótulos de texto em NaN para a coluna ficar totalmente numérica
df['income_dkw'] = (
    df['income_dkw']
    .astype('string')
    .str.replace(' ', '', regex=False)
    .str.replace('Ar', '', regex=False)
    # your code here: also remove commas (.str.replace),
    #                 then replace the labels 'unknown' and 'NA' with np.nan (.replace)
    # o seu código aqui: retire também as vírgulas (.str.replace)
    #                    e depois substitua os rótulos 'unknown' e 'NA' por np.nan (.replace)
)
# Every label is now NaN, so the strict default conversion can be used
# Todos os rótulos são agora NaN, por isso pode usar-se a conversão estrita por omissão
df['income_dkw'] = pd.to_numeric( # your code here: the column, errors='raise' | o seu código aqui: a coluna, errors='raise' )

df['income_dkw'].describe()

**Questions:**

- Which values end up as `NaN` after conversion, and what was each raw value?
- Why replace the text labels (`unknown`, `NA`) with `NaN` *before* converting, and why is the strict default `errors='raise'` a good choice once you have?
- `-5000` and `999999` survived as real numbers. Should they be left as-is here? (They are handled as *coded missing values* in 2.2.3.)

***Perguntas:***

- *Que valores ficam como `NaN` depois da conversão e qual era o valor bruto de cada um?*
- *Porquê substituir os rótulos de texto (`unknown`, `NA`) por `NaN` *antes* de converter e porque é que o valor estrito por omissão `errors='raise'` é uma boa escolha depois disso?*
- *`-5000` e `999999` sobreviveram como números reais. Devem ficar como estão nesta fase? (São tratados como *códigos de valor em falta* no 2.2.3.)*

---

## Task 4: Convert text to dates
*Tarefa 4: Converter texto em datas*

Dates stored as text block any date analysis. A few values use inconsistent formats. Fix those known values with a whole-value `.replace()`, then parse everything with `pd.to_datetime()`. Once every malformed value is fixed or set to `NaN`, use the strict default `errors='raise'`. Use the `.dt` accessor to extract parts.

*Datas guardadas como texto impedem qualquer análise temporal. Alguns valores usam formatos inconsistentes. Corrija esses valores conhecidos com `.replace()` sobre o valor inteiro e depois converta tudo com `pd.to_datetime()`. Quando todos os valores mal formatados estiverem corrigidos ou definidos como `NaN`, use o valor estrito por omissão `errors='raise'`. Use o acessor `.dt` para extrair partes da data.*

In [ ]:
# Inspect the distinct raw dates | Inspecionar as datas brutas distintas
df['survey_date'].unique()

In [ ]:
# Known bad values -> corrected ISO strings; 'not recorded' -> NaN
# Valores incorretos conhecidos -> texto ISO corrigido; 'not recorded' -> NaN
date_fixes = {
    'not recorded': np.nan,
    '03/15/2025':   '2025-03-15',   # MM/DD/YYYY | MM/DD/AAAA
    '2025/01/18':   '2025-01-18',   # slash separators | separadores com barra
    '2025-13-01':   '2025-01-13',   # day/month inverted | dia/mês invertidos
}
df['survey_date'] = df['survey_date'].replace(date_fixes)

# Every malformed value is fixed or NaN, so the strict default conversion can be used
# Todos os valores mal formatados estão corrigidos ou são NaN, por isso pode usar-se a conversão estrita
df['survey_date'] = pd.to_datetime( # your code here: errors='raise' | o seu código aqui: errors='raise' )

df['survey_date'].dtype

In [ ]:
# Extract the month with the .dt accessor (a demonstration, not saved to the checkpoint)
# Extrair o mês com o acessor .dt (demonstração, não é guardada no ponto de controlo)
df['survey_date'].dt.  # your code here: month | o seu código aqui: month


**Questions:**

- Which raw date values needed fixing? What was wrong with each?
- What is `NaT`, and how does it differ from `NaN`?
- Try `df['survey_date'].dt.day_name()`. What other `.dt` properties are useful?

***Perguntas:***

- *Que datas brutas precisaram de correção? O que estava errado em cada uma?*
- *O que é `NaT` e em que difere de `NaN`?*
- *Experimente `df['survey_date'].dt.day_name()`. Que outras propriedades `.dt` são úteis?*

---

## Task 5: Subsetting to inspect problems
*Tarefa 5: Criar subconjuntos para inspecionar problemas*

Filtering lets you *inspect* specific rows without changing the data (in 2.3 you filter to *remove* rows). Use boolean indexing, combine conditions with `&` `|` `~` (wrap each in parentheses), and search text with `isin()` and `str.contains()`.

*A filtragem permite *inspecionar* linhas específicas sem alterar os dados (no 2.3 vai filtrar para *remover* linhas). Use indexação booleana, combine condições com `&` `|` `~` (cada uma entre parênteses) e procure texto com `isin()` e `str.contains()`.*

In [ ]:
# Households with negative income (a likely data error)
# Agregados familiares com rendimento negativo (provável erro de dados)
df[df['income_dkw'] < 0][['hh_id', 'income_dkw']]

In [ ]:
# Combine conditions: rural households with high income
# Combinar condições: agregados rurais com rendimento elevado
df[(df['urban_rural'] == 'Rural') & (df['income_dkw'] > 50000)][['hh_id', 'urban_rural', 'income_dkw']]

In [ ]:
# Filter by a set of districts with isin()
# Filtrar por um conjunto de distritos com isin()
target_districts = ['Polaris District', 'North Delta']
df[df['district'].isin( # your code here | o seu código aqui )][['hh_id', 'district']]

In [ ]:
# Case-insensitive text search, safe with missing values
# Procura de texto sem distinguir maiúsculas de minúsculas, segura com valores em falta
df[df['district'].str.contains( # your code here: 'delta', case=False, na=False | o seu código aqui )][['hh_id', 'district']]

**Questions:**

- How many households have negative income? Which household(s)?
- Why must each condition be wrapped in parentheses when combining with `&`?
- What does `na=False` do in `str.contains()`? What happens without it?

***Perguntas:***

- *Quantos agregados familiares têm rendimento negativo? Quais?*
- *Porque é que cada condição tem de estar entre parênteses quando se combina com `&`?*
- *O que faz `na=False` em `str.contains()`? O que acontece sem esse argumento?*

---

## Task 6: Save a typed checkpoint to `10_cleaned/`
*Tarefa 6: Guardar um ponto de controlo com tipos corrigidos em `10_cleaned/`*

Types are now fixed. Save a **typed checkpoint** so the cleaning step (2.2.3) can start from a stable baseline instead of re-reading raw. Keep the core columns with their corrected types: derived columns (like a `survey_month`) belong to the feature step (2.2.4), so we don't persist them here.

*Os tipos já estão corrigidos. Guarde um **ponto de controlo com tipos corrigidos** para que o passo de limpeza (2.2.3) possa partir de uma base estável em vez de voltar a ler os dados brutos. Mantenha as colunas principais com os tipos corrigidos: as colunas derivadas (como um `survey_month`) pertencem ao passo de criação de variáveis (2.2.4), por isso não as guardamos aqui.*

> **Never** modify files in `0_raw/`. Write to `10_cleaned/`.

> ***Nunca** modifique ficheiros em `0_raw/`. Escreva em `10_cleaned/`.*

In [ ]:
# Keep the core columns for the checkpoint | Manter as colunas principais para o ponto de controlo
COLS_OUT = [
    'hh_id', 'region_code', 'province_name', 'district', 'urban_rural',
    'hh_size', 'income_dkw', 'survey_date', 'pop_density', 'education_code', 'age',
]
df_typed = df[COLS_OUT].copy()
print('Typed checkpoint:', df_typed.shape)
df_typed.head()

In [ ]:
DATA_CLEAN_DIR = '../../data/10_cleaned'
os.makedirs(DATA_CLEAN_DIR, exist_ok=True)
out_path = os.path.join(DATA_CLEAN_DIR, 'datania_households_clean.csv')

df_typed.to_csv( # your code here: index=False | o seu código aqui: index=False )
print('Saved:', out_path)

In [ ]:
# Reload to confirm it round-trips | Voltar a carregar para confirmar que o ficheiro está correto
check = pd.read_csv(out_path, dtype={'hh_id': str, 'region_code': str})
print('Reloaded:', check.shape)
print(check.dtypes)
check.head()

**Questions:**

- After reloading, what dtype does `survey_date` have? What does that tell you about CSV?
- Why `index=False`?
- This file is a *typed checkpoint*, not the final cleaned data. What still has to happen in 2.2.3?

***Perguntas:***

- *Depois de voltar a carregar, que dtype tem `survey_date`? O que é que isso lhe diz sobre o formato CSV?*
- *Porquê `index=False`?*
- *Este ficheiro é um *ponto de controlo com tipos corrigidos*, não os dados finais limpos. O que falta ainda fazer no 2.2.3?*